# Small Object Detection in Aerial Imagery

This notebook demonstrates tuning SAHI's slicing parameters for an aerial scene and exporting results as COCO JSON for downstream evaluation or labeling review.

Key parameters:
- **`slice_height` / `slice_width`**: Should be close to the model's native input size (640 px for YOLO)
- **`overlap_height_ratio` / `overlap_width_ratio`**: Controls how much adjacent tiles share. 0.2 (20%) is a good default; increase to 0.4 if objects span tile boundaries.
- **`postprocess_type`**: NMS (default) or GREEDYNMM for finer merging control
- **`postprocess_match_threshold`**: IoU threshold for merging overlapping boxes across tiles

## 1. Setup

In [ ]:
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

url = "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/coco_utils/terrain2_coco.jpg"
image_path = "sample_aerial.jpg"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolov8n.pt",
    confidence_threshold=0.25,
    device="cpu",
)

## 2. Parameter Sweep: Slice Size Effect

In [ ]:
slice_sizes = [256, 512, 640]
results = {}

for size in slice_sizes:
    result = get_sliced_prediction(
        image=image_path,
        detection_model=model,
        slice_height=size,
        slice_width=size,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0,
    )
    n = len(result.object_prediction_list)
    results[size] = (result, n)
    print(f"slice_size={size}: {n} detections")

In [ ]:
fig, axes = plt.subplots(1, len(slice_sizes), figsize=(7 * len(slice_sizes), 6))
for ax, size in zip(axes, slice_sizes):
    result, n = results[size]
    result.export_visuals(export_dir=".", file_name=f"result_{size}")
    ax.imshow(mpimg.imread(f"result_{size}.png"))
    ax.set_title(f"slice={size}px  ({n} detections)")
    ax.axis("off")
plt.suptitle("Effect of slice size on detection count")
plt.tight_layout()
plt.show()

## 3. Overlap Parameter Effect

In [ ]:
overlaps = [0.0, 0.2, 0.4]

for overlap in overlaps:
    result = get_sliced_prediction(
        image=image_path,
        detection_model=model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=overlap,
        overlap_width_ratio=overlap,
        verbose=0,
    )
    print(f"overlap={overlap:.0%}: {len(result.object_prediction_list)} detections")

## 4. Export Detections as COCO JSON

In [ ]:
best_result, _ = results[512]

# Export in COCO format for use in annotation tools or evaluation
coco_json = best_result.to_coco_annotations()
print(f"COCO annotations: {len(coco_json)} objects")
print("Sample annotation:")
if coco_json:
    ann = coco_json[0]
    print(f"  category_id: {ann['category_id']}")
    print(f"  bbox: {ann['bbox']}")
    print(f"  score: {ann.get('score', 'N/A')}")

In [ ]:
import json

with open("detections.json", "w") as f:
    json.dump(coco_json, f, indent=2)
print("Saved detections to detections.json")

## 5. Confidence Threshold Effect

Lowering the threshold finds more objects but increases false positives. A threshold of 0.25–0.35 is typical for aerial imagery.

In [ ]:
thresholds = [0.1, 0.25, 0.5]

for thresh in thresholds:
    m = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path="yolov8n.pt",
        confidence_threshold=thresh,
        device="cpu",
    )
    r = get_sliced_prediction(
        image=image_path,
        detection_model=m,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0,
    )
    print(f"confidence_threshold={thresh}: {len(r.object_prediction_list)} detections")